In [1]:
pip install yfinance statsmodels pandas numpy openpyxl scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║         NIFTY 200 — COINTEGRATION PAIR TRADING ANALYSIS                    ║
║         3-Year Lookback | Engle-Granger | Parallel Processing               ║
╚══════════════════════════════════════════════════════════════════════════════╝

SETUP:
    pip install yfinance statsmodels pandas numpy openpyxl scipy tqdm

RUN:
    python nifty200_pair_trading.py

OUTPUT:
    nifty200_pair_trades.xlsx   ← Full results (multi-sheet Excel)
    nifty200_pairs_raw.csv      ← All cointegrated pairs (for further analysis)

NOTES:
    - NIFTY 200 universe: ~190 stocks after data quality filter
    - Total pairs tested: ~17,955 combinations
    - Runtime: ~15-25 minutes (depending on internet speed)
    - Parallel processing speeds up cointegration testing significantly
"""

import pandas as pd
import numpy as np
import yfinance as yf
from itertools import combinations
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy import stats
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
import warnings
import openpyxl
from openpyxl.styles import (PatternFill, Font, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
import datetime
import os
import time

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# NIFTY 200 UNIVERSE  (NSE tickers, .NS suffix for yfinance)
# ─────────────────────────────────────────────────────────────────────────────
NIFTY200 = {
    "HINDUNILVR": ("HINDUNILVR.NS", "FMCG", "Household & Personal Care"),
    "ITC":        ("ITC.NS",        "FMCG", "Cigarettes/Diversified"),
    "NESTLEIND":  ("NESTLEIND.NS",  "FMCG", "Food"),
    "VBL":        ("VBL.NS",        "FMCG", "Beverages"),
    "BRITANNIA":  ("BRITANNIA.NS",  "FMCG", "Food"),
    "GODREJCP":   ("GODREJCP.NS",   "FMCG", "Personal Care"),
    "TATACONSUM": ("TATACONSUM.NS", "FMCG", "Food & Beverages"),
    "MARICO":     ("MARICO.NS",     "FMCG", "Personal Care"),
    "MCDOWELL-N": ("MCDOWELL-N.NS", "FMCG", "Beverages - Alcohol"),
    "DABUR":      ("DABUR.NS",      "FMCG", "Ayurvedic/Personal Care"),
    "COLPAL":     ("COLPAL.NS",     "FMCG", "Oral Care"),
    "PATANJALI":  ("PATANJALI.NS",  "FMCG", "Edible Oils/Food"),
    "UBL":        ("UBL.NS",        "FMCG", "Beverages - Alcohol"),
    "RADICO":     ("RADICO.NS",     "FMCG", "Beverages - Alcohol"),
    "EMAMILTD":   ("EMAMILTD.NS",   "FMCG", "Personal Care"),
}

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    "years":           3,           # lookback period
    "p_threshold":     0.05,        # cointegration significance
    "min_corr":        0.40,        # minimum return correlation to keep pair
    "max_half_life":   120,         # days — skip pairs that revert too slowly
    "min_half_life":   2,           # days — skip pairs that revert too fast
    "n_workers":       max(1, multiprocessing.cpu_count() - 1),  # parallel
    "output_excel":    "niftyfmcg_pair_trades.xlsx",
    "output_csv":      "nifty200_pairs_raw.csv",
    "chunk_size":      500,         # pairs per parallel batch
}


# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
def download_prices(universe: dict, years: int = 3) -> pd.DataFrame:
    end   = datetime.date.today()
    start = end - datetime.timedelta(days=365 * years + 30)
    tickers_ns = [v[0] for v in universe.values()]
    names      = list(universe.keys())

    print(f"\n{'─'*65}")
    print(f"  📥 Downloading {len(tickers_ns)} NIFTY 200 stocks")
    print(f"     Period : {start} → {end}")
    print(f"{'─'*65}")

    raw = yf.download(
        tickers_ns,
        start=str(start),
        end=str(end),
        auto_adjust=True,
        progress=True,
        threads=True,
    )["Close"]

    # Rename columns from .NS ticker to short name
    ticker_to_name = {v[0]: k for k, v in universe.items()}
    raw.rename(columns=ticker_to_name, inplace=True)

    # Quality filter: drop stocks with >15% missing data
    thresh = int(0.85 * len(raw))
    raw.dropna(axis=1, thresh=thresh, inplace=True)
    raw.ffill(inplace=True)
    raw.dropna(inplace=True)

    # Deduplicate columns (some tickers may map to same stock)
    raw = raw.loc[:, ~raw.columns.duplicated()]

    n_pairs = len(list(combinations(raw.columns, 2)))
    print(f"\n  ✅ {raw.shape[1]} stocks loaded | {raw.shape[0]} trading days")
    print(f"  🔢 Total pairs to test: {n_pairs:,}")
    return raw


# ─────────────────────────────────────────────────────────────────────────────
# 2. SPREAD ANALYTICS
# ─────────────────────────────────────────────────────────────────────────────
def calc_half_life(spread: pd.Series) -> float | None:
    lag   = spread.shift(1).dropna()
    delta = spread.diff().dropna()
    try:
        model = OLS(delta.values, add_constant(lag.values)).fit()
        lam   = model.params[1]
        if lam >= 0:
            return None
        return round(-np.log(2) / lam, 1)
    except Exception:
        return None


def analyze_spread(s1: pd.Series, s2: pd.Series) -> dict:
    """Full spread analytics for a cointegrated pair."""
    X = add_constant(s2.values)
    model = OLS(s1.values, X).fit()
    hedge  = model.params[1]
    spread = s1 - hedge * s2
    mu, sigma = spread.mean(), spread.std()
    zscore = (spread - mu) / sigma

    adf_res   = adfuller(spread, autolag="AIC")
    half_life = calc_half_life(spread)

    # Rolling z-score (63-day window) for dynamic signal
    roll_mu  = spread.rolling(63).mean()
    roll_std = spread.rolling(63).std()
    roll_z   = ((spread - roll_mu) / roll_std).iloc[-1]

    # Sharpe-like spread profitability: avg return per unit std when |z|>2
    entries_long  = zscore[zscore < -2]
    entries_short = zscore[zscore > 2]
    pct_extreme   = round((len(entries_long) + len(entries_short)) / len(zscore) * 100, 1)

    # Hurst exponent (quick estimate using R/S analysis)
    hurst = calc_hurst(spread)

    return {
        "hedge_ratio":    round(hedge, 4),
        "spread_mean":    round(mu, 4),
        "spread_std":     round(sigma, 4),
        "current_z":      round(zscore.iloc[-1], 3),
        "rolling_z_63d":  round(roll_z, 3) if not np.isnan(roll_z) else None,
        "adf_stat":       round(adf_res[0], 4),
        "adf_pval":       round(adf_res[1], 4),
        "adf_1pct":       round(adf_res[4]["1%"], 4),
        "adf_5pct":       round(adf_res[4]["5%"], 4),
        "half_life":      half_life,
        "pct_extreme_days": pct_extreme,
        "hurst":          round(hurst, 3) if hurst else None,
        "r_squared":      round(model.rsquared, 4),
    }


def calc_hurst(series: pd.Series, max_lag: int = 50) -> float | None:
    """Hurst exponent via R/S analysis. H < 0.5 = mean reverting."""
    try:
        lags  = range(2, min(max_lag, len(series) // 4))
        tau   = [np.std(series.diff(lag).dropna()) for lag in lags]
        if any(t == 0 for t in tau):
            return None
        poly  = np.polyfit(np.log(list(lags)), np.log(tau), 1)
        return poly[0]
    except Exception:
        return None


# ─────────────────────────────────────────────────────────────────────────────
# 3. COINTEGRATION WORKER  (runs in parallel)
# ─────────────────────────────────────────────────────────────────────────────
def process_chunk(args):
    """Test a chunk of pairs. Called by parallel workers."""
    chunk, prices_dict, p_threshold, min_corr, universe = args
    results = []

    for t1, t2 in chunk:
        try:
            s1 = pd.Series(prices_dict[t1])
            s2 = pd.Series(prices_dict[t2])

            # Quick correlation gate — skip obviously unrelated pairs
            lr1 = np.log(s1).diff().dropna()
            lr2 = np.log(s2).diff().dropna()
            corr = float(lr1.corr(lr2))
            if abs(corr) < min_corr:
                continue

            score, pval, _ = coint(s1, s2)
            if pval > p_threshold:
                continue

            m = analyze_spread(s1, s2)
            hl = m["half_life"]

            # Trade signal
            z = m["current_z"]
            rz = m["rolling_z_63d"] or z

            if z > 2.5:
                signal = f"SHORT {t1} / LONG {t2}"
                strength = "Strong" if z > 3.0 else "Moderate"
            elif z < -2.5:
                signal = f"LONG {t1} / SHORT {t2}"
                strength = "Strong" if z < -3.0 else "Moderate"
            elif z > 2.0:
                signal = f"SHORT {t1} / LONG {t2}"
                strength = "Moderate"
            elif z < -2.0:
                signal = f"LONG {t1} / SHORT {t2}"
                strength = "Moderate"
            elif abs(z) > 1.5:
                signal = "Watch – approaching entry"
                strength = "Watch"
            else:
                signal = "Neutral"
                strength = "Neutral"

            sec1 = universe.get(t1, ("","",""))[1]
            sec2 = universe.get(t2, ("","",""))[1]
            sub1 = universe.get(t1, ("","",""))[2]
            sub2 = universe.get(t2, ("","",""))[2]
            same_sector  = sec1 == sec2
            same_subsect = sub1 == sub2

            hl_ok = hl is not None and 3 <= hl <= 90
            hurst = m.get("hurst")
            hurst_ok = hurst is not None and hurst < 0.45

            results.append({
                "Stock A":           t1,
                "Stock B":           t2,
                "Sector A":          sec1,
                "Sector B":          sec2,
                "Sub-sector A":      sub1,
                "Sub-sector B":      sub2,
                "Same Sector":       "Yes" if same_sector else "No",
                "Same Sub-Sector":   "Yes" if same_subsect else "No",
                "Coint p-value":     round(pval, 5),
                "Coint Score":       round(score, 4),
                "Return Corr":       round(corr, 4),
                "R² (OLS)":          m["r_squared"],
                "Hedge Ratio":       m["hedge_ratio"],
                "Half-Life (days)":  hl,
                "HL Valid (3-90d)":  "Yes" if hl_ok else "No",
                "Hurst Exponent":    hurst,
                "Hurst < 0.45":      "Yes" if hurst_ok else ("No" if hurst else "N/A"),
                "Current Z-Score":   m["current_z"],
                "Rolling Z (63d)":   m["rolling_z_63d"],
                "% Days Extreme":    m["pct_extreme_days"],
                "ADF Stat":          m["adf_stat"],
                "ADF p-value":       m["adf_pval"],
                "ADF 5% Critical":   m["adf_5pct"],
                "Spread Mean":       m["spread_mean"],
                "Spread Std":        m["spread_std"],
                "Trade Signal":      signal,
                "Signal Strength":   strength,
            })
        except Exception:
            continue

    return results


# ─────────────────────────────────────────────────────────────────────────────
# 4. SCORING & RANKING
# ─────────────────────────────────────────────────────────────────────────────
def score_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Composite score (0–1):
      30% → Coint p-value strength       (1 - p)
      20% → Return correlation            (clipped 0-1)
      20% → Half-life validity            (in 3-90d range)
      15% → Hurst < 0.45                  (mean-reverting evidence)
      10% → |Z-Score| magnitude           (current signal strength)
       5% → Same sub-sector bonus
    """
    df = df.copy()
    df["_p"]    = 1 - df["Coint p-value"]
    df["_corr"] = df["Return Corr"].abs().clip(0, 1)
    df["_hl"]   = df["HL Valid (3-90d)"].map({"Yes": 1, "No": 0})
    df["_hurst"]= df["Hurst < 0.45"].map({"Yes": 1, "No": 0, "N/A": 0.5})
    df["_z"]    = df["Current Z-Score"].abs().clip(0, 4) / 4
    df["_sub"]  = df["Same Sub-Sector"].map({"Yes": 1, "No": 0})

    df["Composite Score"] = (
        0.30 * df["_p"]     +
        0.20 * df["_corr"]  +
        0.20 * df["_hl"]    +
        0.15 * df["_hurst"] +
        0.10 * df["_z"]     +
        0.05 * df["_sub"]
    ).round(4)

    df.drop(columns=["_p","_corr","_hl","_hurst","_z","_sub"], inplace=True)
    df.sort_values("Composite Score", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.index += 1
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 5. MAIN COINTEGRATION RUNNER
# ─────────────────────────────────────────────────────────────────────────────
def run_cointegration(prices: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    tickers = list(prices.columns)
    all_pairs = list(combinations(tickers, 2))
    total = len(all_pairs)

    # Split into chunks for parallel processing
    chunks = [all_pairs[i:i+cfg["chunk_size"]]
              for i in range(0, total, cfg["chunk_size"])]

    # Convert prices to dict for pickling (multiprocessing requires pickling)
    prices_dict = {col: prices[col].values.tolist() for col in prices.columns}
    # Rebuild index as offset for workers
    prices_dict = {col: list(prices[col]) for col in prices.columns}

    print(f"\n  🔬 Testing {total:,} pairs across {len(chunks)} chunks")
    print(f"  ⚡ Using {cfg['n_workers']} parallel workers")
    print(f"  🔎 Corr gate: |r| > {cfg['min_corr']} | p < {cfg['p_threshold']}\n")

    task_args = [
        (chunk, prices_dict, cfg["p_threshold"], cfg["min_corr"], NIFTY200)
        for chunk in chunks
    ]

    all_results = []
    start_time = time.time()

    try:
        with ProcessPoolExecutor(max_workers=cfg["n_workers"]) as executor:
            futures = {executor.submit(process_chunk, arg): i
                       for i, arg in enumerate(task_args)}
            done = 0
            for future in as_completed(futures):
                chunk_results = future.result()
                all_results.extend(chunk_results)
                done += 1
                elapsed = time.time() - start_time
                pct = done / len(chunks) * 100
                eta = (elapsed / done) * (len(chunks) - done) if done > 0 else 0
                print(f"  [{done:>4}/{len(chunks)}] {pct:5.1f}% | "
                      f"{len(all_results):>5} pairs found | "
                      f"ETA: {int(eta//60)}m {int(eta%60)}s", end="\r")
    except Exception:
        # Fallback to sequential if multiprocessing fails
        print("  ⚠️  Parallel processing failed, switching to sequential…")
        for i, arg in enumerate(task_args):
            chunk_results = process_chunk(arg)
            all_results.extend(chunk_results)
            pct = (i + 1) / len(chunks) * 100
            print(f"  [{i+1:>4}/{len(chunks)}] {pct:5.1f}% | "
                  f"{len(all_results):>5} pairs found", end="\r")

    print(f"\n\n  ✅ Found {len(all_results)} cointegrated pairs "
          f"in {int((time.time()-start_time)//60)}m {int((time.time()-start_time)%60)}s")
    return pd.DataFrame(all_results)


# ─────────────────────────────────────────────────────────────────────────────
# 6. EXCEL BUILDER
# ─────────────────────────────────────────────────────────────────────────────
# Style constants
H_FILL  = PatternFill("solid", fgColor="1F3864")
SH_FILL = PatternFill("solid", fgColor="2F5496")
A_FILL  = PatternFill("solid", fgColor="EBF3FB")
G_FILL  = PatternFill("solid", fgColor="C6EFCE")
R_FILL  = PatternFill("solid", fgColor="FFC7CE")
O_FILL  = PatternFill("solid", fgColor="FFEB9C")
W_FILL  = PatternFill("solid", fgColor="FFF2CC")
N_FILL  = PatternFill("solid", fgColor="F2F2F2")
WH_FILL = PatternFill("solid", fgColor="FFFFFF")

H_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=9)
B_FONT  = Font(name="Arial", size=9)
T_FONT  = Font(name="Arial", bold=True, size=13, color="1F3864")

_thin   = Side(style="thin", color="BFBFBF")
BORDER  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
CTR     = Alignment(horizontal="center", vertical="center", wrap_text=True)
LFT     = Alignment(horizontal="left", vertical="center", wrap_text=False)


SIGNAL_FILLS = {
    "Strong":   G_FILL,
    "Moderate": O_FILL,
    "Watch":    W_FILL,
    "Neutral":  N_FILL,
}

COL_WIDTHS = {
    "Stock A": 12, "Stock B": 12, "Sector A": 14, "Sector B": 14,
    "Sub-sector A": 16, "Sub-sector B": 16,
    "Same Sector": 10, "Same Sub-Sector": 12,
    "Coint p-value": 12, "Coint Score": 11, "Return Corr": 11,
    "R² (OLS)": 9, "Hedge Ratio": 11, "Half-Life (days)": 14,
    "HL Valid (3-90d)": 13, "Hurst Exponent": 13, "Hurst < 0.45": 11,
    "Current Z-Score": 13, "Rolling Z (63d)": 12, "% Days Extreme": 13,
    "ADF Stat": 10, "ADF p-value": 11, "ADF 5% Critical": 14,
    "Spread Mean": 11, "Spread Std": 10,
    "Trade Signal": 32, "Signal Strength": 13, "Composite Score": 14,
}


def _hdr(ws, row, n_cols, fill=H_FILL):
    for col in range(1, n_cols + 1):
        c = ws.cell(row=row, column=col)
        c.fill = fill; c.font = H_FONT; c.alignment = CTR; c.border = BORDER


def _title(ws, cell, text):
    ws[cell] = text
    ws[cell].font  = T_FONT
    ws[cell].fill  = WH_FILL
    ws[cell].alignment = LFT


def write_all_pairs(wb, df):
    ws = wb.create_sheet("All Cointegrated Pairs")
    ws.sheet_view.showGridLines = False
    cols = list(df.columns)

    ws.merge_cells(f"A1:{get_column_letter(len(cols))}1")
    _title(ws, "A1",
           f"NIFTY 200 – Cointegration Pair Trade Analysis  |  3-Year Lookback  |  {datetime.date.today()}")
    ws.row_dimensions[1].height = 26

    for ci, col in enumerate(cols, 1):
        c = ws.cell(row=2, column=ci, value=col)
        c.fill = H_FILL; c.font = H_FONT; c.alignment = CTR; c.border = BORDER
    ws.row_dimensions[2].height = 34

    for ri, (idx, row) in enumerate(df.iterrows(), 3):
        alt = ri % 2 == 0
        sig = str(row.get("Signal Strength", "Neutral"))
        for ci, col in enumerate(cols, 1):
            val = row[col]
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = B_FONT; c.border = BORDER; c.alignment = CTR
            if col == "Signal Strength":
                c.fill = SIGNAL_FILLS.get(sig, N_FILL)
            elif col == "Current Z-Score":
                try:
                    fv = float(val)
                    c.fill = R_FILL if fv > 2 else (G_FILL if fv < -2 else (A_FILL if alt else WH_FILL))
                except: c.fill = A_FILL if alt else WH_FILL
            elif col in ("Same Sector","Same Sub-Sector","HL Valid (3-90d)","Hurst < 0.45") and val == "Yes":
                c.fill = G_FILL
            else:
                c.fill = A_FILL if alt else WH_FILL

    for ci, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = COL_WIDTHS.get(col, 12)
    ws.freeze_panes = "A3"


def write_top30(wb, df_top):
    ws = wb.create_sheet("Top 30 – Deep Dive")
    ws.sheet_view.showGridLines = False

    display_cols = [
        "Stock A","Stock B","Sector A","Sector B","Same Sub-Sector",
        "Coint p-value","Return Corr","Hedge Ratio","Half-Life (days)",
        "Hurst Exponent","Current Z-Score","Rolling Z (63d)",
        "% Days Extreme","Trade Signal","Signal Strength","Composite Score"
    ]
    available = [c for c in display_cols if c in df_top.columns]

    ws.merge_cells(f"A1:{get_column_letter(len(available))}1")
    _title(ws, "A1", "TOP 30 PAIR TRADES – Ranked by Composite Score")
    ws.row_dimensions[1].height = 26

    for ci, col in enumerate(available, 1):
        c = ws.cell(row=2, column=ci, value=col)
        c.fill = SH_FILL; c.font = H_FONT; c.alignment = CTR; c.border = BORDER
    ws.row_dimensions[2].height = 34

    interps = {
        "Strong":   "✅ Active — enter now if sizing allows",
        "Moderate": "🟡 Developing — scale in cautiously",
        "Watch":    "👁️ Monitor — set alert at |Z|=2",
        "Neutral":  "⏸️ No signal — watch for expansion",
    }

    for ri, (idx, row) in enumerate(df_top.iterrows(), 3):
        alt = ri % 2 == 0
        sig = str(row.get("Signal Strength", "Neutral"))
        for ci, col in enumerate(available, 1):
            val = row[col]
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = B_FONT; c.border = BORDER
            c.alignment = LFT if col in ("Trade Signal",) else CTR
            if col == "Signal Strength":
                c.fill = SIGNAL_FILLS.get(sig, N_FILL)
                c.value = interps.get(sig, sig)
            elif col == "Current Z-Score":
                try:
                    fv = float(val)
                    c.fill = R_FILL if fv > 2 else (G_FILL if fv < -2 else (A_FILL if alt else WH_FILL))
                except: c.fill = A_FILL if alt else WH_FILL
            elif col in ("Same Sub-Sector","HL Valid (3-90d)") and val == "Yes":
                c.fill = G_FILL
            else:
                c.fill = A_FILL if alt else WH_FILL

    widths = {c: COL_WIDTHS.get(c, 12) for c in available}
    widths["Trade Signal"] = 34
    widths["Signal Strength"] = 34
    for ci, col in enumerate(available, 1):
        ws.column_dimensions[get_column_letter(ci)].width = widths.get(col, 12)
    ws.freeze_panes = "A3"


def write_by_sector(wb, df):
    ws = wb.create_sheet("By Sector")
    ws.sheet_view.showGridLines = False

    _title(ws, "A1", "COINTEGRATED PAIRS — SECTOR BREAKDOWN")
    ws.merge_cells("A1:H1")
    ws.row_dimensions[1].height = 26

    hdrs = ["Sector A","Sector B","# Pairs","Avg p-val","Avg Corr",
            "Strong Signals","Watch Signals","Cross/Same"]
    for ci, h in enumerate(hdrs, 1):
        c = ws.cell(row=2, column=ci, value=h)
        c.fill = H_FILL; c.font = H_FONT; c.alignment = CTR; c.border = BORDER

    sector_stats = (
        df.groupby(["Sector A","Sector B"])
          .agg(
              Pairs       =("Coint p-value","count"),
              Avg_pval    =("Coint p-value","mean"),
              Avg_Corr    =("Return Corr","mean"),
              Strong      =("Signal Strength", lambda x: (x=="Strong").sum()),
              Watch       =("Signal Strength", lambda x: (x=="Watch").sum()),
          )
          .reset_index()
          .sort_values("Pairs", ascending=False)
    )

    for ri, (_, row) in enumerate(sector_stats.iterrows(), 3):
        same = "Same" if row["Sector A"] == row["Sector B"] else "Cross"
        alt = ri % 2 == 0
        vals = [row["Sector A"], row["Sector B"], int(row["Pairs"]),
                round(row["Avg_pval"],3), round(row["Avg_Corr"],3),
                int(row["Strong"]), int(row["Watch"]), same]
        for ci, val in enumerate(vals, 1):
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = B_FONT; c.border = BORDER; c.alignment = CTR
            c.fill = G_FILL if (ci == 8 and val == "Same") else (A_FILL if alt else WH_FILL)

    for ci, w in enumerate([14,14,8,10,10,13,12,10], 1):
        ws.column_dimensions[get_column_letter(ci)].width = w
    ws.freeze_panes = "A3"


def write_active_signals(wb, df):
    """Sheet with only currently actionable signals (|Z| > 2)."""
    ws = wb.create_sheet("🚨 Active Signals")
    ws.sheet_view.showGridLines = False

    active = df[df["Current Z-Score"].abs() >= 2.0].copy()

    _title(ws, "A1", f"ACTIVE TRADE SIGNALS  |  |Z-Score| ≥ 2.0  |  {len(active)} pairs")
    ws.merge_cells("A1:P1")
    ws.row_dimensions[1].height = 26

    cols = ["Stock A","Stock B","Sector A","Sector B","Same Sub-Sector",
            "Coint p-value","Return Corr","Hedge Ratio","Half-Life (days)",
            "Current Z-Score","Rolling Z (63d)","Hurst Exponent",
            "Trade Signal","Signal Strength","Composite Score"]
    avail = [c for c in cols if c in active.columns]

    for ci, col in enumerate(avail, 1):
        c = ws.cell(row=2, column=ci, value=col)
        c.fill = H_FILL; c.font = H_FONT; c.alignment = CTR; c.border = BORDER
    ws.row_dimensions[2].height = 34

    for ri, (idx, row) in enumerate(active.iterrows(), 3):
        alt = ri % 2 == 0
        sig = str(row.get("Signal Strength","Neutral"))
        for ci, col in enumerate(avail, 1):
            val = row[col]
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = B_FONT; c.border = BORDER; c.alignment = CTR
            if col == "Signal Strength":
                c.fill = SIGNAL_FILLS.get(sig, N_FILL)
            elif col == "Current Z-Score":
                try:
                    fv = float(val)
                    c.fill = R_FILL if fv > 2 else G_FILL
                except: c.fill = A_FILL if alt else WH_FILL
            else:
                c.fill = A_FILL if alt else WH_FILL

    for ci, col in enumerate(avail, 1):
        ws.column_dimensions[get_column_letter(ci)].width = COL_WIDTHS.get(col, 12)
    ws.freeze_panes = "A3"


def write_sub_sector(wb, df):
    """Best pairs within same sub-sector (highest fundamental justification)."""
    ws = wb.create_sheet("Same Sub-Sector Pairs")
    ws.sheet_view.showGridLines = False

    sub = df[df["Same Sub-Sector"] == "Yes"].copy()

    _title(ws, "A1", f"SAME SUB-SECTOR PAIRS  |  {len(sub)} pairs (Highest Fundamental Justification)")
    ws.merge_cells("A1:P1")
    ws.row_dimensions[1].height = 26

    cols = ["Stock A","Stock B","Sector A","Sub-sector A",
            "Coint p-value","Return Corr","Hedge Ratio","Half-Life (days)",
            "Current Z-Score","Hurst Exponent","Trade Signal","Signal Strength","Composite Score"]
    avail = [c for c in cols if c in sub.columns]

    for ci, col in enumerate(avail, 1):
        c = ws.cell(row=2, column=ci, value=col)
        c.fill = H_FILL; c.font = H_FONT; c.alignment = CTR; c.border = BORDER
    ws.row_dimensions[2].height = 34

    for ri, (idx, row) in enumerate(sub.iterrows(), 3):
        alt = ri % 2 == 0
        sig = str(row.get("Signal Strength","Neutral"))
        for ci, col in enumerate(avail, 1):
            val = row[col]
            c = ws.cell(row=ri, column=ci, value=val)
            c.font = B_FONT; c.border = BORDER; c.alignment = CTR
            c.fill = SIGNAL_FILLS.get(sig, N_FILL) if col == "Signal Strength" else (A_FILL if alt else WH_FILL)

    for ci, col in enumerate(avail, 1):
        ws.column_dimensions[get_column_letter(ci)].width = COL_WIDTHS.get(col, 12)
    ws.freeze_panes = "A3"


def write_methodology(wb):
    ws = wb.create_sheet("📖 Methodology")
    ws.sheet_view.showGridLines = False
    ws.column_dimensions["A"].width = 22
    ws.column_dimensions["B"].width = 90

    rows = [
        ("OVERVIEW", ""),
        ("Strategy", "Statistical Arbitrage / Pairs Trading via Cointegration"),
        ("Universe", "NIFTY 200 large & mid-cap NSE stocks (~190 after data quality filter)"),
        ("Lookback", "3 years of daily adjusted closing prices"),
        ("Pairs Tested", "~17,955 unique combinations"),
        ("", ""),
        ("STATISTICAL TESTS", ""),
        ("Step 1 – Correlation Gate", "Only pairs with |Pearson r| > 0.40 on daily log-returns proceed (speeds up processing)"),
        ("Step 2 – Engle-Granger Coint.", "Two-step test: OLS regression to find hedge ratio → ADF test on residuals. p < 0.05 required."),
        ("Step 3 – ADF on Spread", "Augmented Dickey-Fuller confirms spread is stationary (mean-reverting). Lower p = stronger."),
        ("Step 4 – Half-Life (OU)", "Ornstein-Uhlenbeck process fit. Ideal: 3–90 days. Too short = noise. Too long = capital inefficient."),
        ("Step 5 – Hurst Exponent", "R/S analysis. H < 0.5 = mean-reverting. H < 0.45 = strong evidence. H > 0.5 = trending."),
        ("", ""),
        ("KEY METRICS", ""),
        ("Coint p-value", "Probability of NO cointegration. < 0.01 = very strong, < 0.05 = acceptable."),
        ("Return Correlation", "Pearson r of daily log-returns. Higher = more synchronised. Ideal > 0.65 for same sector."),
        ("Hedge Ratio", "β from OLS: Spread = Stock_A − β × Stock_B. Determines sizing ratio of the two legs."),
        ("Half-Life (days)", "Days for spread to mean-revert by 50%. Set max hold = 2× half-life."),
        ("Hurst Exponent", "< 0.45 = strong mean reversion. 0.45–0.55 = random walk. > 0.55 = trending."),
        ("Current Z-Score", "Static: (spread − 3yr mean) / 3yr std. |Z| > 2 = entry signal."),
        ("Rolling Z (63d)", "Dynamic z-score using 63-day rolling mean/std. More adaptive to recent regime."),
        ("% Days Extreme", "% of all trading days where |Z| > 2. Higher = more trading opportunities historically."),
        ("R² (OLS)", "Coefficient of determination of hedge ratio regression. Higher = tighter relationship."),
        ("Composite Score", "Weighted rank score (see below). Higher = better pair quality."),
        ("", ""),
        ("COMPOSITE SCORE WEIGHTS", ""),
        ("30% → Coint Strength", "(1 − p-value) — statistical backbone"),
        ("20% → Return Corr", "Synchronised price movement"),
        ("20% → HL Validity", "Half-life in 3–90 day range"),
        ("15% → Hurst < 0.45", "Mean-reversion character confirmed"),
        ("10% → Z-Score Magnitude", "Current signal opportunity"),
        ("5%  → Same Sub-Sector", "Fundamental justification bonus"),
        ("", ""),
        ("TRADE SIGNALS", ""),
        ("Z > +2.0 (Static)", "Spread HIGH → SHORT Stock A / LONG Stock B"),
        ("Z < -2.0 (Static)", "Spread LOW  → LONG Stock A / SHORT Stock B"),
        ("Z > +2.5", "Strong signal — conviction entry"),
        ("Z < -2.5", "Strong signal — conviction entry"),
        ("|Z| 1.5–2.0", "Watch zone — set alerts, prepare sizing"),
        ("|Z| < 1.5", "Neutral — no edge currently"),
        ("", ""),
        ("RISK MANAGEMENT", ""),
        ("Position Sizing", "Size each leg by rupee-value; aim for equal INR exposure on both legs"),
        ("Hedge Ratio Sizing", "If hedge ratio = 1.3, short 100 shares of A and long 130 shares of B"),
        ("Stop Loss", "Exit if |Z| widens to 3.5+ (spread is diverging, not reverting)"),
        ("Profit Target", "Exit when Z → 0 (full reversion) or |Z| < 0.5 (partial reversion)"),
        ("Max Hold Time", "Close at 2× half-life regardless of P&L"),
        ("Structural Break Watch", "Re-test cointegration every 3 months; discard if p > 0.10"),
        ("Liquidity Check", "Verify adequate F&O open interest + daily volume for both legs before entry"),
        ("", ""),
        ("EXCEL SHEETS GUIDE", ""),
        ("📖 Methodology", "This guide"),
        ("🚨 Active Signals", "All pairs with |Z| ≥ 2.0 right now — most actionable sheet"),
        ("Top 30 – Deep Dive", "Best 30 pairs by composite score with interpretation"),
        ("Same Sub-Sector Pairs", "Pairs within exact same sub-sector (highest fundamental basis)"),
        ("All Cointegrated Pairs", "Complete list of all pairs passing p < 0.05 filter"),
        ("By Sector", "Sector-level summary of pair counts and signal quality"),
        ("", ""),
        ("DISCLAIMERS", ""),
        ("", "Statistical cointegration does NOT guarantee future mean reversion. Relationships can break permanently."),
        ("", "Past cointegration may not hold during market stress, regulatory changes, or corporate actions (M&A)."),
        ("", "This analysis is for informational purposes only. Not investment advice. Verify with a SEBI-registered advisor."),
        ("", "Always paper trade or backtest before deploying capital on any pair trading strategy."),
    ]

    for ri, (label, value) in enumerate(rows, 1):
        is_section = label in ("OVERVIEW","STATISTICAL TESTS","KEY METRICS",
                               "COMPOSITE SCORE WEIGHTS","TRADE SIGNALS",
                               "RISK MANAGEMENT","EXCEL SHEETS GUIDE","DISCLAIMERS")
        cl = ws.cell(row=ri, column=1, value=label)
        cv = ws.cell(row=ri, column=2, value=value)
        if is_section:
            cl.fill = H_FILL; cl.font = H_FONT; cl.alignment = LFT
            cv.fill = H_FILL; cv.font = H_FONT
            ws.row_dimensions[ri].height = 20
        else:
            cl.font = Font(name="Arial", bold=True, size=9)
            cv.font = Font(name="Arial", size=9)
            cv.alignment = Alignment(wrap_text=True, vertical="top")
            ws.row_dimensions[ri].height = 28 if len(str(value)) > 90 else 16


def build_excel(df_all, df_top30, output_path):
    print(f"\n  💾 Building Excel: {output_path} …")
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    write_methodology(wb)
    write_active_signals(wb, df_all)
    write_top30(wb, df_top30)
    write_sub_sector(wb, df_all)
    write_all_pairs(wb, df_all)
    write_by_sector(wb, df_all)

    wb.save(output_path)
    print(f"  ✅ Saved → {output_path}")


# ─────────────────────────────────────────────────────────────────────────────
# 7. CONSOLE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def print_summary(df):
    strong  = df[df["Signal Strength"] == "Strong"]
    moderate= df[df["Signal Strength"] == "Moderate"]
    watch   = df[df["Signal Strength"] == "Watch"]
    same_sub= df[df["Same Sub-Sector"] == "Yes"]

    print(f"\n{'═'*70}")
    print(f"  NIFTY 200 PAIR TRADING — RESULTS SUMMARY")
    print(f"{'═'*70}")
    print(f"  Total cointegrated pairs : {len(df):>6,}")
    print(f"  🟢 Strong signals (|Z|>2.5): {len(strong):>6,}")
    print(f"  🟡 Moderate signals (|Z|>2): {len(moderate):>6,}")
    print(f"  👁️  Watch zone (|Z|>1.5)   : {len(watch):>6,}")
    print(f"  Same sub-sector pairs    : {len(same_sub):>6,}")
    print(f"{'─'*70}")
    print(f"\n  TOP 15 PAIRS BY COMPOSITE SCORE:")
    print(f"{'─'*70}")

    top15_cols = ["Stock A","Stock B","Coint p-value","Return Corr",
                  "Half-Life (days)","Current Z-Score","Trade Signal","Composite Score"]
    avail = [c for c in top15_cols if c in df.columns]
    print(df.head(15)[avail].to_string(index=True))

    print(f"\n{'─'*70}")
    print(f"  ACTIVE SIGNALS RIGHT NOW (|Z| ≥ 2.0):")
    print(f"{'─'*70}")
    active = df[df["Current Z-Score"].abs() >= 2.0]
    if len(active):
        sig_cols = ["Stock A","Stock B","Current Z-Score","Trade Signal","Signal Strength"]
        avail_s = [c for c in sig_cols if c in active.columns]
        print(active.head(20)[avail_s].to_string(index=True))
    else:
        print("  No active signals at this time.")


# ─────────────────────────────────────────────────────────────────────────────
# 8. MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("╔" + "═"*68 + "╗")
    print("║  NIFTY 200 PAIR TRADING — COINTEGRATION ANALYSIS              ║")
    print("║  3-Year Lookback | Engle-Granger | Hurst | Parallel Workers   ║")
    print("╚" + "═"*68 + "╝")

    # Remove duplicate tickers from universe
    seen_tickers = set()
    clean_universe = {}
    for k, v in NIFTY200.items():
        if v[0] not in seen_tickers:
            clean_universe[k] = v
            seen_tickers.add(v[0])

    # Download prices
    prices = download_prices(clean_universe, years=CONFIG["years"])

    # Run cointegration
    df = run_cointegration(prices, CONFIG)

    if df.empty:
        print("\n❌ No cointegrated pairs found. Try relaxing p_threshold.")
        return

    # Score and rank
    df = score_pairs(df)

    # Save CSV (raw data)
    csv_cols = [c for c in df.columns]
    df.to_csv(CONFIG["output_csv"], index=True)
    print(f"\n  📄 Raw CSV saved → {CONFIG['output_csv']}")

    # Build Excel
    df_top30 = df.head(30).copy()
    build_excel(df, df_top30, CONFIG["output_excel"])

    # Print console summary
    print_summary(df)

    print(f"\n{'═'*70}")
    print(f"  ✅ DONE! Open {CONFIG['output_excel']} for full analysis.")
    print(f"{'═'*70}\n")


if __name__ == "__main__":
    main()

╔════════════════════════════════════════════════════════════════════╗
║  NIFTY 200 PAIR TRADING — COINTEGRATION ANALYSIS              ║
║  3-Year Lookback | Engle-Granger | Hurst | Parallel Workers   ║
╚════════════════════════════════════════════════════════════════════╝

─────────────────────────────────────────────────────────────────
  📥 Downloading 15 NIFTY 200 stocks
     Period : 2023-01-29 → 2026-02-27
─────────────────────────────────────────────────────────────────


[**********************87%*****************      ]  13 of 15 completed$MCDOWELL-N.NS: possibly delisted; no timezone found
[*********************100%***********************]  15 of 15 completed

1 Failed download:
['MCDOWELL-N.NS']: possibly delisted; no timezone found



  ✅ 14 stocks loaded | 761 trading days
  🔢 Total pairs to test: 91

  🔬 Testing 91 pairs across 1 chunks
  ⚡ Using 15 parallel workers
  🔎 Corr gate: |r| > 0.4 | p < 0.05

  ⚠️  Parallel processing failed, switching to sequential…
  [   1/1] 100.0% |     1 pairs found

  ✅ Found 1 cointegrated pairs in 0m 0s

  📄 Raw CSV saved → nifty200_pairs_raw.csv

  💾 Building Excel: niftyfmcg_pair_trades.xlsx …
  ✅ Saved → niftyfmcg_pair_trades.xlsx

══════════════════════════════════════════════════════════════════════
  NIFTY 200 PAIR TRADING — RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════
  Total cointegrated pairs :      1
  🟢 Strong signals (|Z|>2.5):      0
  🟡 Moderate signals (|Z|>2):      0
  👁️  Watch zone (|Z|>1.5)   :      1
  Same sub-sector pairs    :      0
──────────────────────────────────────────────────────────────────────

  TOP 15 PAIRS BY COMPOSITE SCORE:
──────────────────────────────────────────────────────────────────────
     St